# Translation Inference

[video](https://www.youtube.com/watch?v=ISNdQcPhsts)

In [13]:
from pathlib import Path

import torch
from dotenv import load_dotenv
from tokenizers import Tokenizer

import models.deep_learning.architectures.transformer.tasks.translation as trn

load_dotenv()
device = trn.get_device()
device

device(type='cpu')

## Config

In [14]:
CONFIG = trn.Config(
    batch_size=8,
    num_epochs=50,
    lr=1e-4,
    src_seq_len=350,
    tgt_seq_len=350,
    d_model=512,
    dropout=0.1,
    datasource="Helsinki-NLP/opus_books",
    src_lang="en",
    tgt_lang="es",
    model_basename="tmodel_",
    preload="latest",
)

In [15]:
checkpoint_path = CONFIG.latest_weights_file_path()
src_file = Path(CONFIG.tokenizer_src_file)
tgt_file = Path(CONFIG.tokenizer_tgt_file)

print(f"checkpoint: {checkpoint_path}")
print(f"src tokenizer: {src_file}")
print(f"tgt tokenizer: {tgt_file}")

checkpoint: /lustre/diazgonz/ml-notebook/weights/Helsinki-NLP/opus_books/tmodel__en_es/tmodel_49.pt
src tokenizer: /lustre/diazgonz/ml-notebook/data/cache/Helsinki-NLP/opus_books/tokenizer_en.json
tgt tokenizer: /lustre/diazgonz/ml-notebook/data/cache/Helsinki-NLP/opus_books/tokenizer_es.json


## Load tokenizers

In [16]:
tokenizer_src = Tokenizer.from_file(str(src_file))
tokenizer_tgt = Tokenizer.from_file(str(tgt_file))

src_pad_idx = tokenizer_src.token_to_id("[PAD]")
src_sos_idx = tokenizer_src.token_to_id("[SOS]")
src_eos_idx = tokenizer_src.token_to_id("[EOS]")

## Build model and load checkpoint

In [17]:
model = trn.Translator(
    src_vocab_size=tokenizer_src.get_vocab_size(),
    tgt_vocab_size=tokenizer_tgt.get_vocab_size(),
    dropout=CONFIG.dropout,
    src_max_length=CONFIG.src_seq_len,
    tgt_max_length=CONFIG.tgt_seq_len,
    embed_size=CONFIG.d_model,
).to(device)

if checkpoint_path is None:
    raise ValueError("No checkpoint found. Please run the training notebook first.")
state = torch.load(checkpoint_path, map_location=device)  #
model.load_state_dict(state["model_state_dict"])
model.eval();

## Run dataset samples  
Use random rows from the dataset 

In [18]:
num_examples = 5
sample_gen = torch.Generator().manual_seed(42)

### Load dataset

In [19]:
raw_ds = trn.TranslationHFDataset.load_dataset(
    path=CONFIG.datasource,
    name=f"{CONFIG.src_lang}-{CONFIG.tgt_lang}",
    split="train",
)

# Keep only rows that fit the model max lengths
filtered_ds = raw_ds.filter(
    lambda x: (
        len(tokenizer_src.encode(x["translation"][CONFIG.src_lang]).ids)
        <= CONFIG.src_seq_len - 2
        and len(tokenizer_tgt.encode(x["translation"][CONFIG.tgt_lang]).ids)
        <= CONFIG.tgt_seq_len - 1
    )
)

In [20]:
sample_idx = torch.randperm(len(filtered_ds), generator=sample_gen)[
    :num_examples
].tolist()

for i, idx in enumerate(sample_idx, start=1):
    row = filtered_ds[idx]["translation"]
    src_text = row[CONFIG.src_lang]
    tgt_text = row[CONFIG.tgt_lang]

    pred_text = trn.translate_text(
        model=model,
        text=src_text,
        tokenizer_src=tokenizer_src,
        tokenizer_tgt=tokenizer_tgt,
        src_seq_len=CONFIG.src_seq_len,
        tgt_max_len=CONFIG.tgt_seq_len,
        device=device,
    )

    print(f"EXAMPLE {i}")
    print(f"EN: {src_text}")
    print(f"ES target: {tgt_text}")
    print(f"ES pred:   {pred_text}")
    print("-" * 100)

EXAMPLE 1
EN: "Ah! that is suggestive.
ES target: ––¡Ah, esto es muy sugerente!
ES pred:   ––¡ Ah , esto es muy sugerente !
----------------------------------------------------------------------------------------------------
EXAMPLE 2
EN: They have taken much time, and some thought."
ES target: Deben de haberle costado mucho tiempo.
ES pred:   Deben de haberle costado mucho tiempo .
----------------------------------------------------------------------------------------------------
EXAMPLE 3
EN: It was a pretty expensive joke for them, for it cost them two and thirty pounds."
ES target: La broma les ha salido bastante cara, ya que les ha costado treinta y dos libras.
ES pred:   La broma les ha salido bastante cara , ya que les ha costado treinta y dos libras .
----------------------------------------------------------------------------------------------------
EXAMPLE 4
EN: It was the first time since their union that he had parted from her without a full explanation. On the one hand th

## Run custom translations 

In [21]:
examples = [
    "I like machine learning.",
    "How are you today?",
    "This notebook now runs inference from a saved checkpoint.",
]

for sentence in examples:
    translation = trn.translate_text(
        model=model,
        text=sentence,
        tokenizer_src=tokenizer_src,
        tokenizer_tgt=tokenizer_tgt,
        src_seq_len=CONFIG.src_seq_len,
        tgt_max_len=CONFIG.tgt_seq_len,
        device=device,
    )
    print(f"EN: {sentence}")
    print(f"ES: {translation}")
    print("-" * 80)

EN: I like machine learning.
ES: Me gustan las máquinas .
--------------------------------------------------------------------------------
EN: How are you today?
ES: ¿ Cómo está usted ?
--------------------------------------------------------------------------------
EN: This notebook now runs inference from a saved checkpoint.
ES: Al volver del manantial , me en una nueva tienda .
--------------------------------------------------------------------------------
